# Phase 5 Visual Audit: Qdrant Vector Store & LangGraph Multi-Agent Engine

This thin visual audit notebook inspects the retrieval-augmented generation and multi-agent coordination layer:
1. **Payload-Filtered Qdrant MMR:** Vector search preventing cross-company contamination and duplicate comment spam.
2. **LangGraph Multi-Agent Architecture:** `planner` -> (`sql_analyst` || `qdrant_retriever`) -> `verifier` -> `synthesizer`.
3. **Dual Grounding:** Every executive insight pairs quantitative DuckDB SQL time-series metrics with qualitative employee quotes.
4. **Anti-Hallucination Gate:** Verifier ensures sample size >= 30 and citations match canonical records.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.common.config import EngineConfig
from src.common.azure_storage import StorageManager
from src.rag.qdrant_indexer import QdrantIndexer
from src.rag.retriever import QdrantMMRRetriever
from src.agents.graph import create_research_graph, run_research_query

config = EngineConfig.load()
storage = StorageManager(config)
print("✅ Configuration and storage manager loaded successfully.")

### 1. Inspect Persistent Qdrant Collection & MMR Retrieval

In [ ]:
# Initialize Qdrant indexer and retriever
indexer = QdrantIndexer(config)
retriever = QdrantMMRRetriever(config=config, indexer=indexer)

# Test payload-filtered MMR retrieval for Amazon
query = "RTO mandate 5 days return to office employee reaction"
evidence = retriever.retrieve(
    query=query,
    companies=["Amazon"],
    top_k=5
)

print(f"Retrieved {len(evidence)} evidence chunks for Amazon:")
for i, hit in enumerate(evidence, 1):
    print(f"[{i}] {hit['citation']} (Score: {hit['score']})")
    print(f"    Quote: {hit['key_quote'][:120]}...")
    print(f"    Companies: {hit['companies']} | Themes: {hit['themes']}")

### 2. Compile LangGraph Multi-Agent Research Engine

In [ ]:
app = create_research_graph(config)
print("✅ LangGraph Multi-Agent Research workflow compiled successfully.")

### 3. Query Benchmark 1: Amazon 5-Day RTO & Jassy Ultimatum

In [ ]:
q1 = "How did Andy Jassy's 5-day RTO announcement in September 2024 impact Amazon employee sentiment?"
res1 = run_research_query(app, q1)

print("=" * 60)
print("EXECUTIVE RESEARCH REPORT:")
print("=" * 60)
print(res1["final_report"])
print("\n--- Citations Cited ---")
for c in res1.get("citations", []):
    print(f"• {c['citation']}: {c['key_quote'][:80]}...")

### 4. Query Benchmark 2: Meta 'Year of Efficiency' & Layer Flattening

In [ ]:
q2 = "What was the morale impact of Mark Zuckerberg's 2023 Year of Efficiency and flattening middle management?"
res2 = run_research_query(app, q2)

print("=" * 60)
print("EXECUTIVE RESEARCH REPORT:")
print("=" * 60)
print(res2["final_report"])

### 5. Query Benchmark 3: 2022–2026 Tech Layoffs Difference-in-Differences Analysis

In [ ]:
q3 = "Analyze the causal impact of mass layoffs on tech employee morale using Difference-in-Differences comparing treated firms to control firms."
res3 = run_research_query(app, q3)

print("=" * 60)
print("EXECUTIVE RESEARCH REPORT:")
print("=" * 60)
print(res3["final_report"])

### 6. Node Trace & Multi-Agent Execution State Inspection

In [ ]:
print("Multi-Agent Plan:", res1.get("plan", {}))
print("SQL Quantitative Findings Count:", len(res1.get("sql_findings", {})))
print("Qdrant Qualitative Hits Count:", len(res1.get("qdrant_evidence", [])))
print("Anti-Hallucination Verification Validated:", res1.get("verification_passed", False))
print("Sample Size Evaluated:", res1.get("sample_size", 0))